# Lab type: write
# Course: DS203 — Feature Engineering & Pipelines
# Lesson: Deploying and Monitoring Pipelines in Production
# Task: Implement a production-grade validation wrapper around a trained pipeline. The wrapper must validate schema, detect unseen categorical values, and monitor for out-of-distribution numeric ranges.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib
import warnings

# Suppress sklearn warnings for cleaner output
warnings.filterwarnings('ignore')

In [ ]:
# Step 1: Create and train a sample pipeline

np.random.seed(42)
n_samples = 800

# Training data
X_train = pd.DataFrame({
    'tenure_days': np.random.uniform(30, 2000, n_samples),
    'monthly_spend': np.random.uniform(20, 150, n_samples),
    'support_tickets': np.random.poisson(2, n_samples),
    'plan': np.random.choice(['basic', 'standard', 'premium'], n_samples),
    'region': np.random.choice(['US', 'EU', 'APAC'], n_samples),
})

y_train = (
    (X_train['tenure_days'] < 300).astype(int) * 0.4 +
    (X_train['monthly_spend'] < 50).astype(int) * 0.3 +
    (X_train['support_tickets'] > 4).astype(int) * 0.3 +
    np.random.uniform(0, 0.5, n_samples)
)
y_train = (y_train > 0.5).astype(int)

print(f"Training data shape: {X_train.shape}")
print(f"Numeric ranges in training data:")
for col in ['tenure_days', 'monthly_spend', 'support_tickets']:
    print(f"  {col}: [{X_train[col].min():.1f}, {X_train[col].max():.1f}]")
print(f"Categorical values:")
for col in ['plan', 'region']:
    print(f"  {col}: {sorted(X_train[col].unique())}")

In [ ]:
# Build the pipeline
numeric_cols = ['tenure_days', 'monthly_spend', 'support_tickets']
categorical_cols = ['plan', 'region']
all_cols = numeric_cols + categorical_cols

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

print("Pipeline trained successfully.")

In [ ]:
# Step 2: Implement the validation wrapper

# TODO: Implement the function `predict_with_validation`
# This function should:
# 1. Check that all expected columns are present in X
# 2. Detect any unexpected columns and warn
# 3. Check for unseen categorical values (that the encoder hasn't seen)
# 4. Monitor numeric columns for out-of-distribution values (beyond ±3σ of training)
# 5. Return a results dictionary with predictions, errors, and warnings

def predict_with_validation(
    pipeline,
    X,
    expected_columns,
    categorical_cols,
    numeric_cols,
    X_train_reference,  # Training data for computing statistics
    threshold_std=3,
    pipeline_version="v1"
):
    """
    Predict with input validation, categorical drift detection, and numeric monitoring.
    
    Returns:
        dict with keys: 'version', 'predictions', 'errors', 'warnings'
    """
    results = {
        'version': pipeline_version,
        'predictions': None,
        'errors': [],
        'warnings': [],
    }
    
    # TODO: Implement schema validation
    # Check for missing columns
    # Check for unexpected columns
    # Select only expected columns in the right order
    
    # TODO: Implement categorical drift detection
    # For each categorical column, check if any values are unseen (not in training data)
    # Add warning if unseen values are found
    
    # TODO: Implement numeric monitoring
    # For each numeric column, compute mean and std from X_train_reference
    # Flag values more than threshold_std standard deviations from the mean
    # Add warning if out-of-distribution values are found
    
    # TODO: Make prediction (only if no errors)
    # If there are errors, return results with predictions=None
    # Otherwise, call pipeline.predict() and store in results['predictions']
    
    return results

print("Function stub created. Implement the TODO sections above.")

<details>
<summary>🔑 Model implementation — predict_with_validation</summary>

```python
def predict_with_validation(
    pipeline,
    X,
    expected_columns,
    categorical_cols,
    numeric_cols,
    X_train_reference,
    threshold_std=3,
    pipeline_version="v1"
):
    results = {
        'version': pipeline_version,
        'predictions': None,
        'errors': [],
        'warnings': [],
    }

    # 1. Schema: missing columns
    missing = [c for c in expected_columns if c not in X.columns]
    if missing:
        results['errors'].append(f"Missing required columns: {missing}")

    # 2. Schema: unexpected columns (warn, don't block)
    unexpected = [c for c in X.columns if c not in expected_columns]
    if unexpected:
        results['warnings'].append(f"Unexpected columns ignored: {unexpected}")

    # Stop here if schema errors — can't safely select columns
    if results['errors']:
        return results

    X = X[expected_columns].copy()

    # 3. Categorical drift: unseen values
    for col in categorical_cols:
        seen = set(X_train_reference[col].dropna().unique())
        incoming = set(X[col].dropna().unique())
        unseen = incoming - seen
        if unseen:
            results['warnings'].append(
                f"Unseen values in '{col}': {unseen} (encoder will zero them out)"
            )

    # 4. Numeric OOD monitoring
    for col in numeric_cols:
        mean = X_train_reference[col].mean()
        std  = X_train_reference[col].std()
        ood_mask = (X[col] - mean).abs() > threshold_std * std
        n_ood = ood_mask.sum()
        if n_ood > 0:
            results['warnings'].append(
                f"{n_ood} row(s) in '{col}' are more than {threshold_std}σ from training mean "
                f"(mean={mean:.1f}, std={std:.1f})"
            )

    # 5. Predict (no blocking errors at this point)
    results['predictions'] = pipeline.predict(X)
    return results
```

**Key design decisions:**

- **Missing columns are hard errors** — the pipeline will raise an exception or silently drop the feature; surfacing this as an error before calling `predict` gives callers a clean diagnostic.
- **Unseen categoricals and OOD numerics are warnings, not errors** — `handle_unknown='ignore'` in `OneHotEncoder` already zeros out unseen categories; the model produces a prediction, just one you should flag for review.
- **Unexpected columns are silently dropped** — warn so the caller knows, but don't block the prediction; schema additions upstream are common and shouldn't break serving.
- **`X_train_reference` is used for OOD stats** — these statistics must come from training data, not from a running window of production inputs, or you lose a fixed baseline to compare against.

</details>

In [ ]:
# Step 3: Test your implementation with clean data

# Create some test data that matches the training distribution
X_test_clean = pd.DataFrame({
    'tenure_days': np.random.uniform(30, 2000, 50),
    'monthly_spend': np.random.uniform(20, 150, 50),
    'support_tickets': np.random.poisson(2, 50),
    'plan': np.random.choice(['basic', 'standard', 'premium'], 50),
    'region': np.random.choice(['US', 'EU', 'APAC'], 50),
})

result = predict_with_validation(
    pipeline,
    X_test_clean,
    all_cols,
    categorical_cols,
    numeric_cols,
    X_train,
    pipeline_version='churn_v1'
)

print(f"Result keys: {result.keys()}")
print(f"Errors: {result['errors']}")
print(f"Warnings: {result['warnings']}")
print(f"Predictions shape: {result['predictions'].shape if result['predictions'] is not None else 'None'}")

In [ ]:
# Step 4: Test with a missing column

X_test_missing = X_test_clean.drop('region', axis=1)

result = predict_with_validation(
    pipeline,
    X_test_missing,
    all_cols,
    categorical_cols,
    numeric_cols,
    X_train,
    pipeline_version='churn_v1'
)

print("Test: Missing 'region' column")
print(f"Errors: {result['errors']}")
print(f"Predictions: {result['predictions']}")

# Expected: Error saying 'region' is missing, predictions = None

In [ ]:
# Step 5: Test with an unseen categorical value

X_test_unseen_cat = X_test_clean.copy()
X_test_unseen_cat.loc[0, 'region'] = 'LATAM'  # New region not in training data

result = predict_with_validation(
    pipeline,
    X_test_unseen_cat,
    all_cols,
    categorical_cols,
    numeric_cols,
    X_train,
    pipeline_version='churn_v1'
)

print("Test: Unseen categorical value ('LATAM' in 'region')")
print(f"Errors: {result['errors']}")
print(f"Warnings: {result['warnings']}")
print(f"Predictions made: {result['predictions'] is not None}")

# Expected: Warning about unseen 'region', predictions still made (but for that row, the region will be all zeros)

In [ ]:
# Step 6: Test with out-of-distribution numeric values

X_test_ood = X_test_clean.copy()
X_test_ood.loc[0, 'monthly_spend'] = 500  # Way above the training range (20-150)
X_test_ood.loc[1, 'support_tickets'] = 100  # Way above typical (mean ~2)

result = predict_with_validation(
    pipeline,
    X_test_ood,
    all_cols,
    categorical_cols,
    numeric_cols,
    X_train,
    pipeline_version='churn_v1'
)

print("Test: Out-of-distribution numeric values")
print(f"Errors: {result['errors']}")
print(f"Warnings: {result['warnings']}")
print(f"Predictions made: {result['predictions'] is not None}")

# Expected: Warnings about out-of-distribution values in 'monthly_spend' and 'support_tickets'

In [ ]:
# Step 7: Test with extra unexpected columns

X_test_extra = X_test_clean.copy()
X_test_extra['new_feature'] = 42  # Column not in training

result = predict_with_validation(
    pipeline,
    X_test_extra,
    all_cols,
    categorical_cols,
    numeric_cols,
    X_train,
    pipeline_version='churn_v1'
)

print("Test: Extra unexpected column ('new_feature')")
print(f"Errors: {result['errors']}")
print(f"Warnings: {result['warnings']}")
print(f"Predictions made: {result['predictions'] is not None}")

# Expected: Warning about unexpected column being ignored, predictions still made

## Summary

Your validation wrapper should catch three classes of production problems:

1. **Schema errors** (missing columns) — prevent prediction
2. **Categorical drift** (unseen values) — warn but allow prediction
3. **Numeric drift** (out-of-distribution values) — warn but allow prediction

This approach ensures that production predictions are always auditable: you can trace back to the warnings that were raised and decide whether the prediction should be trusted or escalated to a human.